<a href="https://colab.research.google.com/github/Afrinjahan/machine_learning/blob/main/ML_Lab02_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning — Lab Sheet 02
## Data Cleaning & Preprocessing on a Real Dataset (Titanic Passenger Data)

**Brahmaputra International University | Department of CSE**

**Objectives:**
- Load and inspect a real-world dataset with genuine missing values and mixed data types.
- Detect and handle missing values using appropriate imputation strategies.
- Encode categorical variables using Label Encoding and One-Hot Encoding.
- Apply feature scaling (standardization) to numeric features.
- Split the dataset into training and testing sets for later model building.

**Dataset:** Titanic passenger data (891 real records), sourced from the seaborn dataset repository, originally from Kaggle's *Titanic: Machine Learning from Disaster* competition.

### 0. Setup — Install & Import Libraries

In [ ]:
# Run this cell first (Colab already has these pre-installed, but this ensures versions are consistent)
!pip install -q scikit-learn pandas


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)


### 1. Load and Inspect the Dataset

We load the data directly from a public URL, so this works in Colab without any file upload.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")
print(df.shape)
df.head()


(891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB


In [ ]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


We'll work with a focused subset of 8 columns relevant to survival prediction:

In [ ]:
df_lab = df[['survived','pclass','sex','age','sibsp','parch','fare','embarked']].copy()
df_lab.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


### 2. Detect Missing Values

In [ ]:
df.isnull().sum()


,0
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


**Real finding:** ~177 passengers (19.9%) have no recorded age, and 2 passengers (0.2%) have no recorded port of embarkation. This is genuine missing data from 1912 record-keeping.

### 3. Handle Missing Values

We use **median** imputation for `age` (robust to skew) and **mode** imputation for the categorical `embarked` column.

In [ ]:
median_age = df_lab['age'].median()
mode_embarked = df_lab['embarked'].mode()[0]
print('Median age:', median_age)
print('Mode embarked:', mode_embarked)

df_lab['age'] = df_lab['age'].fillna(median_age)
df_lab['embarked'] = df_lab['embarked'].fillna(mode_embarked)

df_lab.isnull().sum()


Median age: 28.0
Mode embarked: S


,0
survived,0
pclass,0
sex,0
age,0
sibsp,0
parch,0
fare,0
embarked,0


### 4. Encode Categorical Variables

**4.1 Label Encoding** (binary `sex` column):

In [ ]:
le = LabelEncoder()
df_lab['sex_encoded'] = le.fit_transform(df_lab['sex'])
print(dict(zip(le.classes_, le.transform(le.classes_))))


{'female': np.int64(0), 'male': np.int64(1)}


**4.2 One-Hot Encoding** (multi-category `embarked` column — avoids implying a false order among categories):

In [ ]:
df_lab = pd.get_dummies(df_lab, columns=['embarked'], prefix='embarked')
print(df_lab.columns.tolist())
df_lab.head()


['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'sex_encoded', 'embarked_C', 'embarked_Q', 'embarked_S']


,survived,pclass,sex,age,sibsp,parch,fare,sex_encoded,embarked_C,embarked_Q,embarked_S
0,0,3,male,22.0,1,0,7.2500,1,False,False,True
1,1,1,female,38.0,1,0,71.2833,0,True,False,False
2,1,3,female,26.0,0,0,7.9250,0,False,False,True
3,1,1,female,35.0,1,0,53.1000,0,False,False,True
4,0,3,male,35.0,0,0,8.0500,1,False,False,True


**Discussion:** Why Label Encoding for `sex` but One-Hot Encoding for `embarked`? Label encoding assigns an arbitrary numeric order (0, 1, 2...), which is fine for a 2-category variable but misleading for 3+ unordered categories — a model might wrongly infer that `Q` (2) > `C` (0). One-Hot Encoding avoids this.

### 5. Feature Scaling

We apply `StandardScaler` (zero mean, unit variance) since features like `fare` (range 0–512) and `age` (range 0–80) are on very different scales — many ML algorithms (KNN, SVM, gradient-descent-based models) perform poorly or converge slowly without scaling.

In [ ]:
numeric_cols = ['age','fare','sibsp','parch']
print('BEFORE SCALING:')
print(df_lab[numeric_cols].describe().round(2))


BEFORE SCALING:
          age    fare   sibsp   parch
count  891.00  891.00  891.00  891.00
mean    29.36   32.20    0.52    0.38
std     13.02   49.69    1.10    0.81
min      0.42    0.00    0.00    0.00
25%     22.00    7.91    0.00    0.00
50%     28.00   14.45    0.00    0.00
75%     35.00   31.00    1.00    0.00
max     80.00  512.33    8.00    6.00


In [ ]:
scaler = StandardScaler()
df_scaled = df_lab.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df_lab[numeric_cols])

print('AFTER SCALING:')
print(df_scaled[numeric_cols].describe().round(2))


AFTER SCALING:
          age    fare   sibsp   parch
count  891.00  891.00  891.00  891.00
mean     0.00    0.00    0.00    0.00
std      1.00    1.00    1.00    1.00
min     -2.22   -0.65   -0.47   -0.47
25%     -0.57   -0.49   -0.47   -0.47
50%     -0.10   -0.36   -0.47   -0.47
75%      0.43   -0.02    0.43   -0.47
max      3.89    9.67    6.78    6.97


### 6. Train-Test Split

In [ ]:
X = df_scaled.drop(columns=['survived','sex'])
y = df_scaled['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('X_train shape:', X_train.shape, ' X_test shape:', X_test.shape)
print('Train survival rate:', round(y_train.mean(), 3))
print('Test survival rate:', round(y_test.mean(), 3))


X_train shape: (712, 9)  X_test shape: (179, 9)
Train survival rate: 0.383
Test survival rate: 0.385


We used `stratify=y` to preserve the same survival rate (~38%) in both the training and test sets — important for classification problems so both splits represent the overall population.

### 7. Lab Exercise

1. Instead of median imputation for `age`, try mean imputation and group-wise imputation (median age per `pclass`). Compare the resulting distributions with a histogram.
2. The `deck` column (not used in this lab) has 77% missing values. Should it be imputed, or dropped entirely? Justify your answer in 3–4 sentences.
3. Repeat One-Hot Encoding for `pclass` instead of treating it as a plain numeric column. Does treating `pclass` as categorical make conceptual sense? Explain why or why not.
4. Apply `MinMaxScaler` instead of `StandardScaler` on the numeric columns and compare the resulting value ranges.
5. Save your completed notebook and submit it (.ipynb) with all code, outputs, and answers above added as new cells.